# Інтелектуальне збагачення Excel-даних

Фінальний проєкт курсу «Генеративний та агентний ШІ».
Автор: Геннадій Воленбовський.

Система приймає Excel-файл і завдання природною мовою, сама визначає,
яких даних бракує, знаходить їх у відкритих джерелах і дописує колонки.

Порядок демонстрації: два завдання з умови курсу, потім два додаткові,
щоб показати роботу без жодної зміни коду, потім обробка помилок,
масштабованість і звірка з еталоном викладача.

In [1]:
import logging, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
logging.basicConfig(level=logging.WARNING, format='%(levelname)s %(message)s')

from src import process_excel
from scripts.validate_against_reference import compare, detect_alt_column

DATA = Path.cwd().parent / 'Data'
print('готово')

готово


## Завдання 1. Пряма відстань між столицями

План будує модель: вона сама обирає інструмент, вказує тип сутності
для розпізнавання назв і межі правдоподібності. Тут вона ще й помічає,
що поняття відстані має два трактування, і додає окрему колонку для
відстані дорогами.

In [2]:
report = process_excel(
    file_path=DATA / 'Input' / 'capitals.xlsx',
    task_description='знайди пряму відстань між столицями в км для колонки distance',
    show_progress=False,
)

Файл: D:\Documents\Google Drive\AI Project\Study\AI Generative Excel Final Project\Data\Input\capitals.xlsx
Збережено: D:\Documents\Google Drive\AI Project\Study\AI Generative Excel Final Project\Data\Output\capitals_enriched.xlsx
Рядків усього: 10
Заповнено: 10
Не знайдено: 0
Помилок: 0
Пропущено (вже заповнені або порожній вхід): 0
Викликів моделі: 1 (токенів: вхід 2377, вихід 322)
Мережевих запитів: 0 (з кешу: 24)
Час: 7.9 с
Оцінена вартість: 0.0000 USD


In [3]:
print(report.plan.model_dump_json(indent=2, exclude={'reasoning'}))
print('\nЧому саме так:', report.plan.reasoning)

{
  "target_column": "distance",
  "value_type": "number",
  "unit": "km",
  "tool": "geo_distance",
  "input_columns": {
    "from": "Capital_From",
    "to": "Capital_To"
  },
  "wikidata_property": null,
  "wikidata_type": "Q515",
  "search_query_template": "straight-line distance between {Capital_From} and {Capital_To} in km",
  "bounds": {
    "min_value": 0.0,
    "max_value": 20050.0
  },
  "ambiguous_interpretations": [
    "road distance"
  ],
  "extra_columns": {
    "distance_road": "road distance"
  }
}

Чому саме так: Для прямої відстані між столицями найкраще використати географічний розрахунок між двома містами. Додано окрему колонку для можливої альтернативної інтерпретації — відстані автомобільними дорогами.


### Звірка з еталоном викладача

Еталон по столицях внутрішньо суперечливий: сім рядків це пряма
відстань, два це відстань дорогами, один не збігається ні з чим.
Тому нижче видно і те, як збігається основна колонка, і те, як
виглядає картина з урахуванням другого трактування.

In [4]:
result = Path(report.output_path)
reference = DATA / 'Reference' / 'capitals_enriched.xlsx'
compare(result, reference, 'distance', detect_alt_column(result, reference));


Звірка «distance»: capitals_enriched.xlsx проти capitals_enriched.xlsx
 рядок     еталон       наше   відхилення  вердикт
     2       2500       2023       -19.1%  ПОЗА ДОПУСКОМ, але «distance_road» = 2364 збігається
     3       1435       1434        -0.1%  у допуску
     4        878        876        -0.2%  у допуску
     5        779        524       -32.7%  ПОЗА ДОПУСКОМ
     6        688        689         0.1%  у допуску
     7       1363       1363         0.0%  у допуску
     8        502        503         0.2%  у допуску
     9        243        214       -11.9%  ПОЗА ДОПУСКОМ, але «distance_road» = 244 збігається
    10        416        417         0.2%  у допуску
    11        563        561        -0.4%  у допуску

У допуску ±10% по колонці «distance»: 7 з 10 (70%)
Збігається хоч одне з двох трактувань «distance» або «distance_road»: 9 з 10 (90%)


## Завдання 2. Висота гір

Те саме, але значення бере структуроване джерело Wikidata.
Зверніть увагу на рядок K2 у логу: назва неоднозначна, система
позначила це і звернулась по підтвердження.

In [5]:
report = process_excel(
    file_path=DATA / 'Input' / 'mountains.xlsx',
    task_description='додай висоту гір у метрах до колонки height',
    show_progress=False,
)

Файл: D:\Documents\Google Drive\AI Project\Study\AI Generative Excel Final Project\Data\Input\mountains.xlsx
Збережено: D:\Documents\Google Drive\AI Project\Study\AI Generative Excel Final Project\Data\Output\mountains_enriched.xlsx
Рядків усього: 10
Заповнено: 10
Не знайдено: 0
Помилок: 0
Пропущено (вже заповнені або порожній вхід): 0
Викликів моделі: 2 (токенів: вхід 2649, вихід 338)
Мережевих запитів: 0 (з кешу: 10)
Час: 5.4 с
Оцінена вартість: 0.0000 USD


In [6]:
for row in report.rows:
    print(f'{row.row_index:>3} {str(row.value):>10}  рівень {row.confidence_level}  {row.status}')
    if row.confidence_level > 1:
        print('     ', row.message)

  2    8848.86  рівень 1  ok
  3     8611.0  рівень 2  ok
      [wikidata_lookup] Назва «K2» неоднозначна; інші кандидати: 3253 (Q872294); підтверджено через llm_knowledge
  4     8586.0  рівень 1  ok
  5     8516.0  рівень 1  ok
  6     8485.0  рівень 1  ok
  7     8188.0  рівень 1  ok
  8     8167.0  рівень 1  ok
  9     8163.0  рівень 1  ok
 10     8126.0  рівень 1  ok
 11     8091.0  рівень 1  ok


In [7]:
result = Path(report.output_path)
compare(result, DATA / 'Reference' / 'mountains_enriched.xlsx', 'height');


Звірка «height»: mountains_enriched.xlsx проти mountains_enriched.xlsx
 рядок     еталон       наше   відхилення  вердикт
     2       8849       8849        -0.0%  у допуску
     3       8611       8611         0.0%  у допуску
     4       8586       8586         0.0%  у допуску
     5       8516       8516         0.0%  у допуску
     6       8485       8485         0.0%  у допуску
     7       8188       8188         0.0%  у допуску
     8       8167       8167         0.0%  у допуску
     9       8163       8163         0.0%  у допуску
    10       8215       8126        -1.1%  у допуску
    11       8091       8091         0.0%  у допуску

У допуску ±10% по колонці «height»: 10 з 10 (100%)


## Завдання 3. Населення міста

Код не змінювався. Змінився тільки текст завдання, і планувальник
сам обрав іншу властивість Wikidata та створив нову колонку.

In [8]:
report = process_excel(
    file_path=DATA / 'Input' / 'capitals.xlsx',
    task_description='додай населення міста з колонки Capital_From у нову колонку Capital_From_population',
    output_path=DATA / 'Output' / 'capitals_population_demo.xlsx',
    show_progress=False,
)
print('\nІнструмент:', report.plan.tool, report.plan.wikidata_property)

Файл: D:\Documents\Google Drive\AI Project\Study\AI Generative Excel Final Project\Data\Input\capitals.xlsx
Збережено: D:\Documents\Google Drive\AI Project\Study\AI Generative Excel Final Project\Data\Output\capitals_population_demo.xlsx
Рядків усього: 10
Заповнено: 10
Не знайдено: 0
Помилок: 0
Пропущено (вже заповнені або порожній вхід): 0
Викликів моделі: 9 (токенів: вхід 4922, вихід 3371)
Мережевих запитів: 0 (з кешу: 10)
Час: 13.6 с
Оцінена вартість: 0.0000 USD

Інструмент: wikidata_lookup P1082


## Завдання 4. Дата першого сходження

Тут структурованого джерела немає, тому працює загальний шлях:
пошук в інтернеті плюс витягання значення моделлю. Тип значення не
числовий, а дата, тобто допуск ±10 відсотків до нього незастосовний.

In [9]:
report = process_excel(
    file_path=DATA / 'Input' / 'mountains.xlsx',
    task_description='додай дату першого успішного сходження на гору у форматі ДД.ММ.РРРР до нової колонки first_ascent',
    output_path=DATA / 'Output' / 'mountains_first_ascent_demo.xlsx',
    show_progress=False,
)
print('\nІнструмент:', report.plan.tool)
for row in report.rows:
    print(f'{str(row.value):>12}  {row.source_url[:70]}')

Файл: D:\Documents\Google Drive\AI Project\Study\AI Generative Excel Final Project\Data\Input\mountains.xlsx
Збережено: D:\Documents\Google Drive\AI Project\Study\AI Generative Excel Final Project\Data\Output\mountains_first_ascent_demo.xlsx
Рядків усього: 10
Заповнено: 10
Не знайдено: 0
Помилок: 0
Пропущено (вже заповнені або порожній вхід): 0
Викликів моделі: 21 (токенів: вхід 24550, вихід 3145)
Мережевих запитів: 0 (з кешу: 10)
Час: 14.5 с
Оцінена вартість: 0.0000 USD

Інструмент: web_search_extract
  29.05.1953  https://en.wikipedia.org/wiki/Mount_Everest
  31.07.1954  https://www.instagram.com/reel/DRlqb1VgeV7?hl=en
  25.05.1955  https://www.altitudehimalaya.com/blog/kangchenjunga
  18.05.1956  https://en.wikipedia.org/wiki/Lhotse
  15.05.1955  https://globalsummitguide.com/mountains-makalu-climb-guide-nepal-tibet
  19.10.1954  https://explorersweb.com/a-brief-history-of-climbing-on-cho-oyu
  13.05.1960  https://nepalgram.com/trip/dhaulagiri-expedition
  09.05.1956  https://www.ne

## Обробка помилок

Файл `broken.xlsx` зібраний навмисно поламаним. Жоден випадок не
валить прогін: кожен рядок отримує статус і зрозуміле пояснення,
а файл усе одно зберігається.

In [10]:
report = process_excel(
    file_path=DATA / 'Input' / 'broken.xlsx',
    task_description='додай висоту гір у метрах до колонки height',
    show_progress=False,
)
print()
for row in report.rows:
    print(f'{row.row_index:>3} {str(row.value):>10}  {row.status}')
    print('     ', row.message[:150])

Файл: D:\Documents\Google Drive\AI Project\Study\AI Generative Excel Final Project\Data\Input\broken.xlsx
Збережено: D:\Documents\Google Drive\AI Project\Study\AI Generative Excel Final Project\Data\Output\broken_enriched.xlsx
Рядків усього: 6
Заповнено: 1
Не знайдено: 1
Помилок: 2
Пропущено (вже заповнені або порожній вхід): 2
Викликів моделі: 4 (токенів: вхід 7016, вихід 614)
Мережевих запитів: 4 (з кешу: 3)
Час: 10.1 с
Оцінена вартість: 0.0000 USD

  2    8848.86  ok
      [wikidata_lookup] Структуроване джерело Wikidata
  3       None  out_of_bounds
      [wikidata_lookup] Назва «Atlantis» неоднозначна; інші кандидати: 178 (Q3493630): Значення 4 менше за нижню межу 100 | [web_search_extract] Значення з 
  4       None  skipped_empty_input
      У рядку порожня вхідна колонка «Mountain»
  5       None  skipped_filled
      Комірка вже заповнена, оригінальні дані не чіпаємо
  6       None  out_of_bounds
      [wikidata_lookup] Структуроване джерело Wikidata: Значення 21229 більше за 

## Масштабованість

Тисяча пар столиць зі ста міст. Координати беруться пакетами, далі
працює формула, тому мережевих запитів одиниці, а не тисячі.
Дорожній режим вимкнено: публічний OSRM вимагає паузи в секунду між
запитами, це обмеження джерела, а не системи.

In [11]:
report = process_excel(
    file_path=DATA / 'Input' / 'stress_1000.xlsx',
    task_description='знайди пряму відстань між столицями в км для колонки distance',
    plan_path=DATA / 'plans' / 'capitals_distance.json',
    road_mode=False,
    max_workers=16,
    show_progress=False,
)

Файл: D:\Documents\Google Drive\AI Project\Study\AI Generative Excel Final Project\Data\Input\stress_1000.xlsx
Збережено: D:\Documents\Google Drive\AI Project\Study\AI Generative Excel Final Project\Data\Output\stress_1000_enriched.xlsx
Рядків усього: 1000
Заповнено: 1000
Не знайдено: 0
Помилок: 0
Пропущено (вже заповнені або порожній вхід): 0
Викликів моделі: 0 (токенів: вхід 0, вихід 0)
Мережевих запитів: 6 (з кешу: 12)
Час: 9.3 с
Оцінена вартість: 0.0000 USD


## Підсумок

Чотири різні завдання пройшли одним і тим самим кодом: змінювався
тільки текст завдання, а план під нього щоразу будувала модель.

Головне архітектурне рішення проєкту: те, що задається формулою або
структурованим джерелом, рахує код, а моделі лишається тільки те, що
потребує розуміння природної мови. Тому тисяча рядків коштує два
мережевих запити і жодного виклику моделі понад один план на файл.

Розбіжність з еталоном викладача на трьох рядках розібрана в README:
два з них це відстань дорогами, походження третього невідоме.